# DSC PJATK Catalogue Quality Challenge — EDA

Eksploracja danych treningowych, z której wynikły dwie zmiany w modelu:
1. odczyt liczby sztuk z tekstu (`main_items`, porównanie #1),
2. cena i waga za sztukę względem typowej dla produktu (`OffTypical`, porównanie #2).

Target służy tu tylko do sprawdzania hipotez na train. Nic z tego notebooka nie trafia do modelu jako dane.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_colwidth', None)
local = Path('data/public')
data_dir = local if (local / 'train.csv').exists() else next(Path('/kaggle/input').rglob('train.csv')).parent
train = pd.read_csv(data_dir / 'train.csv')
desc = train.description.fillna('').str.lower()
print(train.shape)
train.head()

## Krok 1: wskazówka „liczba sztuk” w baseline

Baseline bierze pierwszą liczbę przed „pcs” i porównuje ją z `pack_size`. Komentarz organizatorów ostrzega przed gratisami i liczbami zapisanymi słowami.

In [ ]:
has_pcs = desc.str.contains(r'\d+\s*pcs')
has_free = desc.str.contains('free accessor')
print('Opisy z "pcs":', has_pcs.sum())
print('...w tym z darmowymi akcesoriami:', (has_pcs & has_free).sum())

In [ ]:
first = pd.to_numeric(desc.str.extract(r'(\d+)\s*pcs', expand=False), errors='coerce')
wrong = first.notna() & train.pack_size.notna() & (first != train.pack_size)
print('Liczba przed "pcs" inna niż pack_size:', wrong.sum())
print('...z tego w opisach z gratisami:', (wrong & has_free).sum())
print(train.loc[wrong & has_free, ['description', 'pack_size']].head(3).to_string())

In [ ]:
words = r'\b(?:one|two|three|four|five)\b'
has_word = desc.str.contains(words)
print('Opisy z liczbą zapisaną słowem:', has_word.sum())
print('...w tym bez "pcs" (baseline nic tu nie odczyta):', (has_word & ~has_pcs).sum())

**Wniosek:** 98% niezgodności to oferty z gratisami (baseline czyta liczbę gratisów), a 3979 ofert ma liczbę tylko słowem.

Hipoteza odrzucona: czy brak marki w tytule oznacza złą ofertę?

In [ ]:
brand = train.brand.fillna('').str.lower()
title = train.title.fillna('').str.lower()
train['marka_w_tytule'] = [b in t for b, t in zip(brand, title)]
train[(brand != '') & (title != '')].groupby('marka_w_tytule').target.agg(['count', 'mean'])

## Krok 2: jakie formuły ilości występują w opisach

Zamiana każdej liczby na `#` sprawia, że zdania z różnymi liczbami wyglądają tak samo; `value_counts` pokazuje listę szablonów.

In [ ]:
num = r'\b(?:\d+|one|two|three|four|five)\b'
zdania = desc.str.split(r'[.;]\s*', regex=True).explode()
z_liczba = zdania[zdania.str.contains(num)]
szablony = z_liczba.str.replace(num, '#', regex=True)
szablony.value_counts().head(15)

In [ ]:
WORDS = {'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5'}
QTY = (r'(?:pack of|contains|main items:|quantity:?)\s*(\d+)'
       r'|(\d+)\s*(?:-pack|units|pcs|main items|individually)')


def main_items(text):
    text = text.lower()
    text = re.sub(r'also includes \w+ pcs of free accessory pieces', ' ', text)
    for word, digit in WORDS.items():
        text = re.sub(r'\b' + word + r'\b', digit, text)
    found = re.search(QTY, text)
    if found is None:
        return np.nan
    return float(found.group(1) or found.group(2))


print(main_items('Also includes 3 pcs of free accessory pieces; these do not count as main items. Pack of five main items.'))
print(main_items('Included main items: 2'))
print(main_items('Sold as a 3-pack'))
print(main_items('6-string small guitar by Mira'))

In [ ]:
text = train.title.fillna('') + ' ' + train.description.fillna('')
new_pack = text.apply(main_items)
diff = (new_pack - train.pack_size).abs()
print('Odczytano liczbę w', new_pack.notna().sum(), 'z', len(train), 'ofert')
print(pd.crosstab(diff > 0, train.target))

**Wniosek:** nowy odczyt różni się od `pack_size` w 0 dobrych i 391 złych ofertach (stary: 2088 dobrych i 499 złych).

## Krok 3: cena i waga względem typowej dla produktu

Nazwa produktu z opisu („Microphone stand by Mira”), potem odległość ceny/wagi za sztukę od mediany produktu (w logarytmach, czyli „ile razy”).
Tu mediany liczone są z całego train tylko do eksploracji; w modelu liczy je `OffTypical.fit` na foldzie treningowym.

In [ ]:
train['product'] = desc.str.extract(r'(?:^|\.\s*)([a-z][a-z \-]+?) by \w+', expand=False)
print('Rozpoznany produkt w', train['product'].notna().sum(), 'z', len(train), 'ofert, różnych produktów:', train['product'].nunique())
print(train.loc[0, ['title', 'product', 'shipping_weight_kg', 'target']])

In [ ]:
for col in ['shipping_weight_kg', 'price_pln']:
    value = np.log(train[col] / train.pack_size.clip(lower=1))
    typical = value.groupby(train['product']).transform('median')
    off = (value - typical).abs()
    ok = off.notna()
    print(col)
    print(train.groupby(pd.cut(off, [0, 0.3, 0.5, 0.7, 1, 10], include_lowest=True), observed=True).target.agg(['count', 'mean']).round(3))
    print('AUC samej wartości na sztukę:', round(roc_auc_score(train.target[ok], value[ok]), 4))
    print('AUC odległości od typowej:   ', round(roc_auc_score(train.target[ok], off[ok]), 4))
    print()

**Wniosek:** sama waga i cena na sztukę mają AUC ok. 0,47 (jak rzut monetą), bo „drogo” i „ciężko” zależą od produktu.
Odległość od mediany produktu daje 0,69 (waga) i 0,70 (cena); przy odległości > 1 (ok. 2,7×) złych jest 100% (waga) i 78% (cena).